# Monthly versus daily backtests: 47-month cumulative returns

The saved monthly-model predictions and RET selections are fixed. First compare the monthly-return backtest with daily cash/share accounting over exactly the same close-to-close boundaries. Then show next-open execution, costs and stops separately. All four Paper15 RF/XGBoost 5/10-year arms remain.

**Every cumulative performance figure spans February 2020–December 2023: 47 months / 985 following sessions. Five/ten years denotes the training window. Annualized CAGR is labelled separately below.**, initial 31 January close. Terminal holdings are marked without forced liquidation. Zero refits or reselection.

[Results and interpretation](../docs/FIXED_FORECAST_RESULTS_20261027.md). Run All authenticates saved evidence and reads reports. No downloads, model loading, training or trade replay.


In [ ]:
from pathlib import Path
import json, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
local = ROOT / "configs/fixed_forecast_reconciliation.local.json"
settings = json.loads(local.read_text()) if local.exists() else {}
FIXED_FORECAST_RUN = Path(settings.get("output_directory", ROOT / "outputs/fixed_forecast_reconciliation/20261027-v1"))
ANCHOR_COMPLETION_RUN = Path(settings.get("anchor_completion_directory", ROOT / "outputs/fixed_forecast_anchor_completion/20261027-v1"))


In [ ]:
from dmv2.fixed_forecast_reporting import read_fixed_forecast_reconciliation
from dmv2.fixed_forecast_completion_reporting import read_fixed_forecast_completed
fixed_saved = None
try:
    fixed_saved = (read_fixed_forecast_completed(FIXED_FORECAST_RUN, ANCHOR_COMPLETION_RUN)
                   if (ANCHOR_COMPLETION_RUN / "ledger.json").exists()
                   else read_fixed_forecast_reconciliation(FIXED_FORECAST_RUN))
except FileNotFoundError:
    print("Private evidence is unavailable. No compute started.")
else:
    print(fixed_saved["status"], "new fits:", fixed_saved["new_fits"])
    if fixed_saved["status"] == "COMPLETE_VERIFIED":
        print(fixed_saved["period"])
        print("Complete:", fixed_saved["complete_paths"], "Withheld:", fixed_saved["withheld_paths"])
        print("Original-cache close controls reconcile:", fixed_saved["all_close_controls_reconcile"])
        if "original_comparison" in fixed_saved:
            print("Preserved original coverage: 22 complete / 6 withheld. One observed PBF boundary completes the six close controls.")
            print("All open-unit controls exactly match the monthly close convention:", fixed_saved["all_completed_controls_reconcile"])
            print("Tiny recorded conversion-unit residuals remain; see the results note. No tolerance was relaxed.")
    else:
        print("Independent results verification is pending.")


## Gross cumulative total returns over 47 months

Gross cumulative returns: monthly-return engine, daily engine with matching close fills, and daily engine with next-open fills. The same baskets are used throughout.


In [ ]:
if fixed_saved and fixed_saved["status"] == "COMPLETE_VERIFIED":
    comp = fixed_saved["comparison"]
    model_names = {"rf_paper15_5y": "RF · 5-year training", "xgb_paper15_5y": "XGBoost · 5-year training",
                   "rf_paper15_10y": "RF · 10-year training", "xgb_paper15_10y": "XGBoost · 10-year training"}
    gross = comp.pivot(index="arm", columns="stage", values="total_return")
    display((100 * gross[["cache_close_gross", "execution_close_gross", "execution_open_gross"]])
            .rename(index=model_names, columns={"cache_close_gross": "Monthly: 47-month gross total (%)",
             "execution_close_gross": "Daily same-close: 47-month gross total (%)", "execution_open_gross": "Daily next-open: 47-month gross total (%)"}).round(4))


## Month-by-month similarity

Compare the 47 monthly-engine returns with daily next-open returns compounded by calendar month. Correlations and percentage-point gaps are gross, without stops. Displayed curves are checked against the independently verified daily NAV ledgers.


In [ ]:
if fixed_saved and fixed_saved["status"] == "COMPLETE_VERIFIED":
    import pandas as pd
    similarity = []
    for arm, g in fixed_saved["monthly"].groupby("arm"):
        p = g.pivot(index="outcome_month", columns="stage", values="return")
        a, b = p["cache_close_gross"], p["execution_open_gross"]
        if a.notna().all() and b.notna().all() and len(p) == 47:
            similarity.append({"Model": model_names[arm], "Paired months": len(p), "Correlation": a.corr(b),
                 "Mean absolute gap (pp)": 100 * (b-a).abs().mean(), "Largest gap (pp)": 100 * (b-a).abs().max(),
                 "Same monthly sign (%)": 100 * a.gt(0).eq(b.gt(0)).mean()})
    display(pd.DataFrame(similarity).round(4))


## Net cumulative total returns over 47 months

One-way costs are 10 bp. The stop variant uses the existing 10% per-asset rule, gap-at-open / idealized daily high-low threshold fills, and keeps stopped capital in cash.


In [ ]:
if fixed_saved and fixed_saved["status"] == "COMPLETE_VERIFIED":
    display((100 * gross[["cache_close_net", "execution_open_net", "execution_open_net_stop"]])
            .rename(index=model_names, columns={"cache_close_net": "Monthly: 47-month net total (%)",
             "execution_open_net": "Daily next-open: 47-month net total (%)",
             "execution_open_net_stop": "Daily + stop: 47-month net total (%)"}).round(4))
    display(fixed_saved["comparison"].rename(columns={"total_return": "cumulative_total_return_47_months_decimal", "cagr": "annualized_CAGR_decimal"}))


## Annualized CAGR, separate from cumulative totals

The same 47-month paths, after 10 bp one-way costs. Annualized geometric CAGR uses the independently verified daily convention `(ending NAV / starting NAV) ** (252 / 985) - 1`. This differs from an annualized arithmetic mean return. All amounts in the table are percentages.


In [ ]:
if fixed_saved and fixed_saved["status"] == "COMPLETE_VERIFIED":
    annualized = []
    for arm, name in model_names.items():
        z = comp.loc[comp["arm"].eq(arm)].set_index("stage")
        annualized.append({"Model / training window": name,
            "No-stop 47-month net total (%)": 100*z.loc["execution_open_net", "total_return"],
            "No-stop annualized CAGR (%)": 100*z.loc["execution_open_net", "cagr"],
            "Stop: 47-month net total (%)": 100*z.loc["execution_open_net_stop", "total_return"],
            "Stop: annualized CAGR (%)": 100*z.loc["execution_open_net_stop", "cagr"]})
    display(pd.DataFrame(annualized).round(2))


## Scope and preserved references

The original 28-path run remains 22 complete / 6 withheld. The additive observed PBF 30 December 2022 boundary completion replays just those six diagnostic close controls; all original 22 complete rows, including next-open/cost/stop results, remain unchanged. Its composite view has 28 complete paths. The original withheld rows remain available below.

Exact close-clock equality does not establish investability or certify every individual training label, source identity, distribution/action history or historical knowledge time. Next-open paths have different fill boundaries; stops change holding paths. Earlier executable-target forecasts and RET baskets differ from the monthly-model forecasts fixed here, so their performance gap is not a pure execution effect. Same-close execution, fractional merger units and zero borrowing/funding remain assumptions. All prior runs, the master and notebooks 10/11/12 remain unchanged; financial 2024+ values stay protected.


In [ ]:
if fixed_saved and fixed_saved["status"] == "COMPLETE_VERIFIED":
    if "original_comparison" in fixed_saved:
        original = fixed_saved["original_comparison"]
        display(original.loc[~original["complete_path"]])
        display(fixed_saved["completion_reference_differences"].groupby(["arm", "stage"])
                .agg(max_absolute_NAV_gap=("absolute_nav_difference", lambda s: s.abs().max()),
                     exact_tolerance_failures=("matched", lambda s: (~s).sum())))
